# 12. Router Knowledge Base — Send questions to the right source

A knowledge-base assistant is only useful if it searches the right source. This example routes questions to policy, product, or troubleshooting knowledge before retrieval.

**Learning goals**
- Define source-specific knowledge boundaries.
- Route questions before searching.
- Evaluate router decisions separately from answer quality.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

## 12.1 Define knowledge sources

Knowledge sources should have clear scope. Separating policy, product, and troubleshooting content reduces irrelevant retrieval.


In [ ]:
knowledge_sources = {
    "billing": ["Refunds are available within 7 days of payment."],
    "technical": ["Error reports require logs and reproduction steps."],
    "product": ["Deep Agents include planning, files, and subagents."],
}

list(knowledge_sources)

## 12.2 Deterministic router

The router chooses where to search. In production this could be model-assisted, but deterministic routing is ideal for learning and testing the contract.


In [ ]:
def route_source(question: str) -> str:
    q = question.lower()
    if "refund" in q or "refund" in q:
        return "billing"
    if "error" in q or "error" in q:
        return "technical"
    return "product"

route_source("What features does Deep Agents provide?")

## 12.3 Source-local search

Once a source is selected, search only inside that source. This makes retrieval behavior easier to debug and evaluate.


In [ ]:
def retrieve(question: str) -> dict:
    source = route_source(question)
    docs = knowledge_sources[source]
    return {"source": source, "documents": docs}

retrieve("Tell me the refund conditions")

## 12.4 Router evaluation

Router evaluation asks a narrow question: did the system choose the right source? Keeping this separate from answer evaluation makes failures easier to diagnose.


In [ ]:
cases = [
    ("Can I get a refund?", "billing"),
    ("The app has an error", "technical"),
    ("What are Deep Agents?", "product"),
]

for question, expected in cases:
    actual = route_source(question)
    print(question, actual, actual == expected)

---

## Summary

| Item | Content |
|---|---|
| **Covered** | source routing, source-local retrieval, routing tests, and knowledge-base specialization |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`custom-multi-agent.md`](../../docs/langchain/multi-agent/custom-workflow.md)
- [`retrieval.md`](../../docs/langchain/retrieval.md)
- [`workflows-agents.md`](../../docs/langgraph/workflows-agents.md)
